In [30]:
import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix

In [31]:
BASE_DIR = os.path.abspath("../dataset/chest_xray")

TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR = os.path.join(BASE_DIR, "val")
TEST_DIR = os.path.join(BASE_DIR, "test")

In [25]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

In [32]:
print("BASE_DIR:", BASE_DIR)
print("TRAIN_DIR exists:", os.path.exists(TRAIN_DIR))
print("VAL_DIR exists:", os.path.exists(VAL_DIR))
print("TEST_DIR exists:", os.path.exists(TEST_DIR))

BASE_DIR: C:\Users\KIIT0001\Documents\pneumonia_project\dataset\chest_xray
TRAIN_DIR exists: True
VAL_DIR exists: True
TEST_DIR exists: True


In [33]:
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20         # Increased for proper convergence

In [34]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.15
)

In [35]:
train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="training"
)

Found 4434 images belonging to 2 classes.


In [7]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.15
)

In [37]:
val_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="validation"
)

Found 782 images belonging to 2 classes.


In [38]:
test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

test_gen = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

Found 624 images belonging to 2 classes.


In [39]:
plt.figure(figsize=(8, 4))
for i in range(6):
    imgs, labels = next(train_gen)
    plt.subplot(2, 3, i + 1)
    plt.imshow((imgs[0] - imgs[0].min()) / (imgs[0].max() - imgs[0].min()))
    plt.title("Pneumonia" if labels[0] == 1 else "Normal")
    plt.axis("off")
plt.tight_layout()
plt.savefig("sample_images.png")

In [40]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False

x = GlobalAveragePooling2D()(base_model.output)
x = Dense(128, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(1, activation="sigmoid")(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=Adam(learning_rate=1e-4),
     loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.Precision(name="precision")
    ]
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [41]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    ModelCheckpoint("pneumonia_mobilenetv2_binary.h5", save_best_only=True),
    ReduceLROnPlateau(monitor="val_loss", patience=2, factor=0.5)
]

In [42]:
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks
)

Epoch 1/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.7786 - auc: 0.7665 - loss: 0.4725 - precision: 0.8101 - recall: 0.9197  

139/139 ━━━━━━━━━━━━━━━━━━━━ 545s 4s/step - accuracy: 0.8333 - auc: 0.8849 - loss: 0.3623 - precision: 0.8552 - recall: 0.9338 - val_accuracy: 0.9220 - val_auc: 0.9723 - val_loss: 0.2128 - val_precision: 0.9467 - val_recall: 0.9484 - learning_rate: 1.0000e-04
Epoch 2/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9198 - auc: 0.9703 - loss: 0.2049 - precision: 0.9351 - recall: 0.9580  

139/139 ━━━━━━━━━━━━━━━━━━━━ 512s 3s/step - accuracy: 0.9204 - auc: 0.9696 - loss: 0.2014 - precision: 0.9362 - recall: 0.9581 - val_accuracy: 0.9284 - val_auc: 0.9768 - val_loss: 0.1800 - val_precision: 0.9597 - val_recall: 0.9432 - learning_rate: 1.0000e-04
Epoch 3/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9287 - auc: 0.9734 - loss: 0.1868 - precision: 0.9494 - recall: 0.9543  

139/139 ━━━━━━━━━━━━━━━━━━━━ 428s 3s/step - accuracy: 0.9292 - auc: 0.9757 - loss: 0.1785 - precision: 0.9474 - recall: 0.9578 - val_accuracy: 0.9271 - val_auc: 0.9798 - val_loss: 0.1686 - val_precision: 0.9596 - val_recall: 0.9415 - learning_rate: 1.0000e-04
Epoch 4/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9405 - auc: 0.9783 - loss: 0.1666 - precision: 0.9607 - recall: 0.9584  

139/139 ━━━━━━━━━━━━━━━━━━━━ 466s 3s/step - accuracy: 0.9411 - auc: 0.9807 - loss: 0.1574 - precision: 0.9597 - recall: 0.9611 - val_accuracy: 0.9488 - val_auc: 0.9838 - val_loss: 0.1447 - val_precision: 0.9704 - val_recall: 0.9604 - learning_rate: 1.0000e-04
Epoch 5/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9487 - auc: 0.9854 - loss: 0.1365 - precision: 0.9694 - recall: 0.9613  

139/139 ━━━━━━━━━━━━━━━━━━━━ 422s 3s/step - accuracy: 0.9477 - auc: 0.9854 - loss: 0.1370 - precision: 0.9665 - recall: 0.9630 - val_accuracy: 0.9514 - val_auc: 0.9851 - val_loss: 0.1391 - val_precision: 0.9772 - val_recall: 0.9570 - learning_rate: 1.0000e-04
Epoch 6/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9494 - auc: 0.9858 - loss: 0.1360 - precision: 0.9647 - recall: 0.9662  

139/139 ━━━━━━━━━━━━━━━━━━━━ 440s 3s/step - accuracy: 0.9504 - auc: 0.9853 - loss: 0.1342 - precision: 0.9669 - recall: 0.9663 - val_accuracy: 0.9591 - val_auc: 0.9907 - val_loss: 0.1179 - val_precision: 0.9824 - val_recall: 0.9621 - learning_rate: 1.0000e-04
Epoch 7/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 407s 3s/step - accuracy: 0.9522 - auc: 0.9872 - loss: 0.1264 - precision: 0.9690 - recall: 0.9666 - val_accuracy: 0.9425 - val_auc: 0.9871 - val_loss: 0.1319 - val_precision: 0.9637 - val_recall: 0.9587 - learning_rate: 1.0000e-04
Epoch 8/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9552 - auc: 0.9890 - loss: 0.1190 - precision: 0.9714 - recall: 0.9679  

139/139 ━━━━━━━━━━━━━━━━━━━━ 427s 3s/step - accuracy: 0.9511 - auc: 0.9876 - loss: 0.1231 - precision: 0.9678 - recall: 0.9663 - val_accuracy: 0.9552 - val_auc: 0.9919 - val_loss: 0.1081 - val_precision: 0.9659 - val_recall: 0.9742 - learning_rate: 1.0000e-04
Epoch 9/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 471s 3s/step - accuracy: 0.9511 - auc: 0.9878 - loss: 0.1238 - precision: 0.9689 - recall: 0.9651 - val_accuracy: 0.9565 - val_auc: 0.9911 - val_loss: 0.1114 - val_precision: 0.9757 - val_recall: 0.9656 - learning_rate: 1.0000e-04
Epoch 10/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 570s 4s/step - accuracy: 0.9571 - auc: 0.9892 - loss: 0.1147 - precision: 0.9717 - recall: 0.9706 - val_accuracy: 0.9488 - val_auc: 0.9895 - val_loss: 0.1190 - val_precision: 0.9788 - val_recall: 0.9518 - learning_rate: 1.0000e-04
Epoch 11/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9582 - auc: 0.9910 - loss: 0.1076 - precision: 0.9730 - recall: 0.9703  

139/139 ━━━━━━━━━━━━━━━━━━━━ 233s 2s/step - accuracy: 0.9569 - auc: 0.9898 - loss: 0.1125 - precision: 0.9726 - recall: 0.9693 - val_accuracy: 0.9668 - val_auc: 0.9930 - val_loss: 0.0979 - val_precision: 0.9809 - val_recall: 0.9742 - learning_rate: 5.0000e-05
Epoch 12/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 212s 2s/step - accuracy: 0.9565 - auc: 0.9905 - loss: 0.1082 - precision: 0.9717 - recall: 0.9696 - val_accuracy: 0.9527 - val_auc: 0.9887 - val_loss: 0.1208 - val_precision: 0.9642 - val_recall: 0.9725 - learning_rate: 5.0000e-05
Epoch 13/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 212s 2s/step - accuracy: 0.9619 - auc: 0.9915 - loss: 0.1035 - precision: 0.9751 - recall: 0.9736 - val_accuracy: 0.9565 - val_auc: 0.9930 - val_loss: 0.1002 - val_precision: 0.9757 - val_recall: 0.9656 - learning_rate: 5.0000e-05
Epoch 14/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 211s 2s/step - accuracy: 0.9583 - auc: 0.9908 - loss: 0.1075 - precision: 0.9718 - recall: 0.9721 - val_accuracy: 0.9540 - val_auc: 0.9893 - val_loss: 0.11

In [45]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Train Accuracy")
plt.plot(history.history["val_accuracy"], label="Val Accuracy")
plt.legend()
plt.title("Accuracy")

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Val Loss")
plt.legend()
plt.title("Loss")

plt.tight_layout()
plt.savefig("training_graph.png")

In [46]:
test_results = model.evaluate(test_gen)
for name, value in zip(model.metrics_names, test_results):
    print(f"{name}: {value:.4f}")

# Predictions and classification report
pred_probs = model.predict(test_gen)
y_pred = (pred_probs > 0.5).astype(int).ravel()
y_true = test_gen.classes

print(classification_report(y_true, y_pred, target_names=["Normal", "Pneumonia"]))

20/20 ━━━━━━━━━━━━━━━━━━━━ 24s 1s/step - accuracy: 0.8397 - auc: 0.9618 - loss: 0.4213 - precision: 0.8008 - recall: 0.9897            
loss: 0.4213
compile_metrics: 0.8397
20/20 ━━━━━━━━━━━━━━━━━━━━ 19s 798ms/step
              precision    recall  f1-score   support

      Normal       0.97      0.59      0.73       234
   Pneumonia       0.80      0.99      0.89       390

    accuracy                           0.84       624
   macro avg       0.89      0.79      0.81       624
weighted avg       0.86      0.84      0.83       624



In [48]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Normal", "Pneumonia"],
    yticklabels=["Normal", "Pneumonia"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.savefig("confusion_matrix.png")